# Islamic AI — Step 4: Export to GGUF (Mobile Format)

This notebook converts the trained model to GGUF format — the format used by `llama.cpp` which runs on mobile devices (Flutter app).

**What this does:**
- Merges the LoRA adapters back into the base model
- Quantizes to Q4_K_M (4-bit) — reduces size from ~6GB to ~1.8GB
- Saves the `.gguf` file to Google Drive

**Requirements:**
- Run notebook 03 first (training must be complete)
- T4 GPU selected

**Estimated time:** 20–30 minutes

## Cell 1 — Install Libraries

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
print('Done.')

## Cell 2 — Mount Drive and Load Trained Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
import os

LORA_PATH = '/content/drive/MyDrive/islamic_ai/model/lora'
GGUF_DIR  = '/content/drive/MyDrive/islamic_ai/model/gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

print(f'Loading trained model from {LORA_PATH} ...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = LORA_PATH,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

print('\n✅ Trained model loaded.')
print('Files in LoRA directory:')
for f in os.listdir(LORA_PATH):
    size = os.path.getsize(f'{LORA_PATH}/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')

## Cell 3 — Export to GGUF (Q4_K_M Quantization)

`Q4_K_M` = 4-bit quantization with K-means optimization.
- File size: ~1.8 GB
- Quality: Excellent — minimal quality loss vs full precision
- Speed: Fast on mobile CPU

This is the recommended format for on-device mobile inference.

In [ ]:
import time

GGUF_FILENAME = 'islamic_ai_q4_k_m.gguf'
GGUF_PATH     = f'/content/{GGUF_FILENAME}'

print('Exporting to GGUF format...')
print('Quantization: Q4_K_M')
print('This takes 15–25 minutes. Please wait...')
print()

start = time.time()

# Unsloth handles the full merge + quantize + export pipeline
model.save_pretrained_gguf(
    '/content/islamic_ai',   # Output directory (filename auto-generated)
    tokenizer,
    quantization_method = 'q4_k_m',
)

elapsed = time.time() - start
print(f'\nExport completed in {elapsed / 60:.1f} minutes.')

# Find the generated GGUF file
import glob
gguf_files = glob.glob('/content/islamic_ai*.gguf') + glob.glob('/content/*.gguf')
if gguf_files:
    gguf_file = gguf_files[0]
    size_gb   = os.path.getsize(gguf_file) / 1e9
    print(f'\n✅ GGUF file created: {gguf_file}')
    print(f'   File size: {size_gb:.2f} GB')
else:
    print('⚠️  GGUF file not found at expected location. Check /content/')
    !ls /content/*.gguf 2>/dev/null || echo 'No .gguf files found'

## Cell 4 — Copy GGUF to Google Drive

In [ ]:
import shutil

DRIVE_GGUF = f'{GGUF_DIR}/islamic_ai_q4_k_m.gguf'

print(f'Copying GGUF to Google Drive...')
print(f'Source      : {gguf_file}')
print(f'Destination : {DRIVE_GGUF}')
print('This may take a few minutes for a 1.8GB file...')

shutil.copy2(gguf_file, DRIVE_GGUF)

final_size = os.path.getsize(DRIVE_GGUF) / 1e9
print(f'\n✅ GGUF saved to Google Drive!')
print(f'   {DRIVE_GGUF} ({final_size:.2f} GB)')

## Cell 5 — Also Export Q8 Version (Higher Quality, Larger Size)

Optional: Export a Q8_0 version (~3.3GB) for better quality on higher-end phones.

In [ ]:
# ─── OPTIONAL — comment out if you only want Q4_K_M ───

print('Exporting Q8_0 version (higher quality, ~3.3GB)...')

model.save_pretrained_gguf(
    '/content/islamic_ai_q8',
    tokenizer,
    quantization_method = 'q8_0',
)

q8_files = glob.glob('/content/islamic_ai_q8*.gguf') + glob.glob('/content/*q8*.gguf')
if q8_files:
    q8_file = q8_files[0]
    q8_size = os.path.getsize(q8_file) / 1e9
    shutil.copy2(q8_file, f'{GGUF_DIR}/islamic_ai_q8_0.gguf')
    print(f'\n✅ Q8_0 GGUF saved: {q8_size:.2f} GB')
    print(f'   Saved to: {GGUF_DIR}/islamic_ai_q8_0.gguf')

print('\nDone. Your GGUF files are in Google Drive → islamic_ai → model → gguf')

## ✅ Export Complete!

Your model is now ready for mobile deployment.

| File | Size | Use Case |
|------|------|----------|
| `islamic_ai_q4_k_m.gguf` | ~1.8 GB | Recommended for most phones |
| `islamic_ai_q8_0.gguf` | ~3.3 GB | Higher-end phones, better quality |

**Next steps:**
1. Download the GGUF file from Google Drive to your computer
2. Run `05_test_model.ipynb` to test the model quality
3. Move to Phase 6 — integrate into the Flutter app

**To download from Drive:**
- Open Google Drive in your browser
- Navigate to `islamic_ai → model → gguf`
- Right-click the `.gguf` file → Download